In [ ]:
import sys
from pathlib import Path

sys.path.append(str(Path.cwd()))

import numpy as np
import pandas as pd
from linearmodels.panel import PanelOLS, RandomEffects, PooledOLS
from scipy.stats import chi2, f

from Module.panel_utils import (
    ModelResultsAggregator,
    run_panel_regressions,
    run_spec_tests,
    run_panel_model_diagnostics,
)


In [52]:
###############################
# ЗАГРУЗКА И ПОДГОТОВКА ДАННЫХ
###############################
df_reg_analys = pd.read_excel('reg_analys.xlsx')
df_fed_analys = pd.read_excel('fed_analys.xlsx')

# Убираем служебные столбцы индекса
df_reg_analys = df_reg_analys.loc[:, ~df_reg_analys.columns.str.startswith('Unnamed')]
df_fed_analys = df_fed_analys.loc[:, ~df_fed_analys.columns.str.startswith('Unnamed')]

# Приводим даты к datetime
df_reg_analys['Date'] = pd.to_datetime(df_reg_analys['Date'])
df_fed_analys['Date'] = pd.to_datetime(df_fed_analys['Date'])
 
# Объединяем региональные и федеральные данные
fed_extra_cols = [
    col for col in df_fed_analys.columns
    if col not in df_reg_analys.columns and col != 'Region'
]
df_reg = df_reg_analys.merge(
    df_fed_analys[['Date'] + fed_extra_cols],
    on='Date',
    how='left'
)

df_reg['Cluster_3'] = ((df_reg['Cluster_1'] == 0) & (df_reg['Cluster_2'] == 0)).astype(int)


# Взаимодействия общего шока
if 'd_Mon_Shock' in df_reg.columns and 'Cluster_1' in df_reg.columns:
    df_reg['d_Mon_Shock_Cl1'] = df_reg['d_Mon_Shock'] * df_reg['Cluster_1']
if 'd_Mon_Shock' in df_reg.columns and 'Cluster_2' in df_reg.columns:
    df_reg['d_Mon_Shock_Cl2'] = df_reg['d_Mon_Shock'] * df_reg['Cluster_2']
if 'd_Mon_Shock' in df_reg.columns and 'Cluster_3' in df_reg.columns:
    df_reg['d_Mon_Shock_Cl3'] = df_reg['d_Mon_Shock'] * df_reg['Cluster_3']
if 'd_Mon_Shock' in df_reg.columns and 'Covid_dum' in df_reg.columns:
    df_reg['d_Mon_Shock_Covid'] = df_reg['d_Mon_Shock'] * df_reg['Covid_dum']
if 'd_Mon_Shock' in df_reg.columns and 'Sank_dum' in df_reg.columns:
    df_reg['d_Mon_Shock_Sank'] = df_reg['d_Mon_Shock'] * df_reg['Sank_dum']


# Взаимодействия негативного шока 
if 'd_Mon_Shock' in df_reg.columns and 'Cluster_1' in df_reg.columns:
    df_reg['neg_d_Mon_Shock_Cl1'] = df_reg['d_Mon_Shock_neg'] * df_reg['Cluster_1']
if 'd_Mon_Shock' in df_reg.columns and 'Cluster_2' in df_reg.columns:
    df_reg['neg_d_Mon_Shock_Cl2'] = df_reg['d_Mon_Shock_neg'] * df_reg['Cluster_2']
if 'd_Mon_Shock' in df_reg.columns and 'Cluster_3' in df_reg.columns:
    df_reg['neg_d_Mon_Shock_Cl3'] = df_reg['d_Mon_Shock_neg'] * df_reg['Cluster_3']
if 'd_Mon_Shock' in df_reg.columns and 'Covid_dum' in df_reg.columns:
    df_reg['neg_d_Mon_Shock_Covid'] = df_reg['d_Mon_Shock_neg'] * df_reg['Covid_dum']
if 'd_Mon_Shock' in df_reg.columns and 'Sank_dum' in df_reg.columns:
    df_reg['neg_d_Mon_Shock_Sank'] = df_reg['d_Mon_Shock_neg'] * df_reg['Sank_dum']

# Взаимодействия позитивного шока 
if 'd_Mon_Shock' in df_reg.columns and 'Cluster_1' in df_reg.columns:
    df_reg['pos_d_Mon_Shock_Cl1'] = df_reg['d_Mon_Shock_pos'] * df_reg['Cluster_1']
if 'd_Mon_Shock' in df_reg.columns and 'Cluster_2' in df_reg.columns:
    df_reg['pos_d_Mon_Shock_Cl2'] = df_reg['d_Mon_Shock_pos'] * df_reg['Cluster_2']
if 'd_Mon_Shock' in df_reg.columns and 'Cluster_3' in df_reg.columns:
    df_reg['pos_d_Mon_Shock_Cl3'] = df_reg['d_Mon_Shock_pos'] * df_reg['Cluster_3']
if 'd_Mon_Shock' in df_reg.columns and 'Covid_dum' in df_reg.columns:
    df_reg['pos_d_Mon_Shock_Covid'] = df_reg['d_Mon_Shock_pos'] * df_reg['Covid_dum']
if 'd_Mon_Shock' in df_reg.columns and 'Sank_dum' in df_reg.columns:
    df_reg['neg_d_Mon_Shock_Sank'] = df_reg['d_Mon_Shock_pos'] * df_reg['Sank_dum']

# Кластерные выборки
if 'Cluster_1' in df_reg.columns and 'Cluster_2' in df_reg.columns:
    df_reg_clus_one = df_reg[df_reg['Cluster_1'] == 1].copy()
    df_reg_clus_two = df_reg[df_reg['Cluster_2'] == 1].copy()
    df_reg_clus_three = df_reg[df_reg['Cluster_3'] == 1].copy()


In [53]:
# # quick column comparison
# reg_cols = df_reg_analys.columns.tolist()
# fed_cols = df_fed_analys.columns.tolist()

# cols_common = [c for c in reg_cols if c in fed_cols]
# cols_only_reg = [c for c in reg_cols if c not in fed_cols]
# cols_only_fed = [c for c in fed_cols if c not in reg_cols]

# cols_common, cols_only_reg, cols_only_fed


### Попытка как у Скуратовой & Зверевой

In [54]:
### Что нужно
# exog_vars_initial = [
#     'd_Mon_Shock',         # Монетарный индикатор
#     'Covid_dum',           # Экономический шок
#     'Sank_dum',            
#     'd_Mon_Shock_Covid',   # Взаимодействие c шоками
#     'd_Mon_Shock_Sank',    
#     'Cluster_1',           # Социальная неоднородность
#     'Cluster_2',           
#     'Cluster_3',           
#     'd_Mon_Shock_Cl1',     # Взаимодействие c социальной неоднородностью (кластеры)
#     'd_Mon_Shock_Cl2',     
#     'd_Mon_Shock_Cl3',     
#     'D_top5_rozn',         # Особенности банковского сектора (пока только топ-5)
#     'd_Inflation_Expectations',   # Федеральные факторы
#     'd_Ex_Rate',
#     'Bonds_Rate_Correct_5Y',
# ]


### Что ещё можно использовать
    
    # 'Cred_nagr',
    # 'D_top5_rozn',
    # 'Fin_Dostup',
    # 'Credit_impulse',
    # 'Cred_structure',
    # 'Def_Zadolg_ConsCred',

In [55]:
# Лаги 
df_reg["d_Mon_Shock_lag1"] = df_reg["d_Mon_Shock"].shift(1)
df_reg["d_Ex_Rate_lag1"] = df_reg["d_Ex_Rate"].shift(1)


In [56]:
###############################
# Построение линейных моделей на панельных данных (КАК В ИССЛЕДОВАНИИ СКУРАТОВА & ЗВЕРЕВА)
# (Int_Rate_ConsCred зависимая переменная)
# Модель в первой разности - первый лаг ДКП
###############################

exog_vars_initial = [
    'd_Mon_Shock_lag1',         # Монетарный индикатор
    'D_top5_rozn',         # Особенности банковского сектора (пока только топ-5)
    'd_Ex_Rate',
    'd_Ex_Rate_lag1',
]

dependent_var = 'd_Int_Rate_ConsCred'

y, X, pooled_res, fe_res, re_res, pooled_success, fe_success, re_success = run_panel_regressions(
    df_reg, df_reg, dependent_var, exog_vars_initial, cov_type='clustered', cluster_entity=True, df_name='df_reg'
)

# Сохраняем результаты модели для экспорта
pooled_res_77 = pooled_res if pooled_success else None
fe_res_77 = fe_res if fe_success else None
re_res_77 = re_res if re_success else None

ПАНЕЛЬНАЯ РЕГРЕССИЯ: POOL + FE + RE (ОБЩАЯ ВЫБОРКА)
Зависимая переменная: d_Int_Rate_ConsCred

МОДЕЛЬ 1: POOLED OLS
                           PooledOLS Estimation Summary                          
Dep. Variable:     d_Int_Rate_ConsCred   R-squared:                        0.1077
Estimator:                   PooledOLS   R-squared (Between):              0.9586
No. Observations:                 6156   R-squared (Within):               0.1018
Date:                 Tue, Dec 30 2025   R-squared (Overall):              0.1077
Time:                         15:11:54   Log-likelihood                -1.272e+04
Cov. Estimator:              Clustered                                           
                                         F-statistic:                      185.64
Entities:                           81   P-value                           0.0000
Avg Obs:                        76.000   Distribution:                  F(4,6152)
Min Obs:                        76.000                          

In [57]:
run_panel_model_diagnostics(
    y, X, pooled_res, fe_res, re_res,
    pooled_success, fe_success, re_success
)



MODEL DIAGNOSTICS
Target: d_Int_Rate_ConsCred
p-value threshold: 0.05

Endogeneity (Durbin-Wu-Hausman, control-function)
H0: regressor is exogenous (p < threshold suggests endogeneity).
        variable    t_stat    p_val
d_Mon_Shock_lag1 -2.158900 0.030897
       d_Ex_Rate  1.987367 0.046926
  d_Ex_Rate_lag1  1.951683 0.051021
     D_top5_rozn  1.950125 0.051207

----------------------------------------------------------------------
MODEL: Pooled OLS
----------------------------------------------------------------------

Heteroskedasticity tests
Breusch-Pagan: stat=38.0136, p=0.000000
White:         stat=88.0699, p=0.000000

Normality tests
Jarque-Bera: stat=134309368.6120, p=0.000000, skew=0.4342, kurt=726.6177
Shapiro-Wilk (n=5000): stat=0.4232, p=0.000000

Autocorrelation tests
Durbin-Watson: 2.6689
Wooldridge AR(1): beta=-1.3390, t=-69.0239, p=0.000000, n=6075

Cross-sectional dependence (Pesaran CD)
CD stat=35.6258, p=0.000000, N=81, T_avg=1.00

---------------------------------

In [58]:
# ===== ТЕСТЫ СПЕЦИФИКАЦИИ =====

run_spec_tests(
    y, X, pooled_res, fe_res, re_res, pooled_success, fe_success, re_success
)



ТЕСТЫ СПЕЦИФИКАЦИИ

 ТЕСТ ХАУСМАНА (FE vs RE)
----------------------------------------------------------------------
H-статистика: 191.2706
p-значение: 0.000000
df: 4

Вывод: p < 0.05 - Используйте FIXED EFFECTS

 ТЕСТ БРЕУША-ПАГАНА (RE vs Pooled)
----------------------------------------------------------------------
LM статистика: 3117.0496
p-значение: 0.000000
N (регионов): 81, T (периодов): 76

Вывод: p < 0.05 - Есть региональные эффекты (используй RE или FE)

 F-ТЕСТ (FE vs Pooled)
----------------------------------------------------------------------
F-статистика: -43.7939
p-значение: 1.000000
df: (80, 6071)

Вывод: p >= 0.05 - Pooled адекватна


(191.27056342664702, 0.0, 3117.0496379364395, 0.0, -43.793915950651524, 1.0)

In [59]:
# ###############################
# # Построение линейных моделей на панельных данных (КАК В ИССЛЕДОВАНИИ СКУРАТОВА & ЗВЕРЕВА)
# # (Int_Rate_ConsCred зависимая переменная)
# # Модель в первой разности - первый лаг ДКП
# # Взаимодействие шоков
# ###############################

# exog_vars_initial = [
#     'Covid_dum',           # Экономический шок
#     'Sank_dum',            
#     'd_Mon_Shock_Covid',   # Взаимодействие c шоками
#     'd_Mon_Shock_Sank',    
#     'Cluster_1',           # Социальная неоднородность
#     'Cluster_2',           
#     'Cluster_3',           
#     'd_Mon_Shock_Cl1',     # Взаимодействие c социальной неоднородностью (кластеры)
#     'd_Mon_Shock_Cl2',     
#     'd_Mon_Shock_Cl3',     
#     'D_top5_rozn',         # Особенности банковского сектора (пока только топ-5)
#     'd_Inflation_Expectations',   # Федеральные факторы
#     'd_Ex_Rate',
#     'Bonds_Rate_Correct_5Y',
# ]

# dependent_var = 'd_Int_Rate_ConsCred'

# y, X, pooled_res, fe_res, re_res, pooled_success, fe_success, re_success = run_panel_regressions(
#     df_reg, df_reg, dependent_var, exog_vars_initial, cov_type='clustered', cluster_entity=True, df_name='df_reg'
# )

# # Сохраняем результаты модели для экспорта
# pooled_res_2 = pooled_res if pooled_success else None
# fe_res_2 = fe_res if fe_success else None
# re_res_2 = re_res if re_success else None

In [60]:
# run_panel_model_diagnostics(
#     y, X, pooled_res, fe_res, re_res,
#     pooled_success, fe_success, re_success
# )


In [61]:
# # ===== ТЕСТЫ СПЕЦИФИКАЦИИ =====

# run_spec_tests(
#     y, X, pooled_res, fe_res, re_res, pooled_success, fe_success, re_success
# )


### Собственные модели

In [62]:
###############################
# Построение линейных моделей на панельных данных (общая выборка)
# (Int_Rate_ConsCred зависимая переменная)
###############################

exog_vars_initial = [
    'Int_Rate_ConsCred_lag1',
    'Cred_nagr',
    'D_top5_rozn',
    'Fin_Dostup',
    'Credit_impulse',
    'Cred_structure',
    'Def_Zadolg_ConsCred',
    'd_Mon_Shock',
    'ln_New_Loans_ConsCred',
    'Bonds_Rate_Correct_5Y',
    'd_Ex_Rate',
    'd_Inflation_Expectations',
    'Covid_dum',
    'Sank_dum'
]

dependent_var = 'd_Int_Rate_ConsCred'

y, X, pooled_res, fe_res, re_res, pooled_success, fe_success, re_success = run_panel_regressions(
    df_reg, df_reg, dependent_var, exog_vars_initial, cov_type='clustered', cluster_entity=True, df_name='df_reg'
)

# Сохраняем результаты модели для экспорта
pooled_res_1 = pooled_res if pooled_success else None
fe_res_1 = fe_res if fe_success else None
re_res_1 = re_res if re_success else None



ПАНЕЛЬНАЯ РЕГРЕССИЯ: POOL + FE + RE (ОБЩАЯ ВЫБОРКА)
Зависимая переменная: d_Int_Rate_ConsCred

МОДЕЛЬ 1: POOLED OLS
                           PooledOLS Estimation Summary                          
Dep. Variable:     d_Int_Rate_ConsCred   R-squared:                        0.2876
Estimator:                   PooledOLS   R-squared (Between):             -0.5340
No. Observations:                 6237   R-squared (Within):               0.2931
Date:                 Tue, Dec 30 2025   R-squared (Overall):              0.2876
Time:                         15:11:55   Log-likelihood                -1.215e+04
Cov. Estimator:              Clustered                                           
                                         F-statistic:                      193.29
Entities:                           81   P-value                           0.0000
Avg Obs:                        77.000   Distribution:                 F(13,6224)
Min Obs:                        77.000                          

In [63]:
run_panel_model_diagnostics(
    y, X, pooled_res, fe_res, re_res,
    pooled_success, fe_success, re_success
)



MODEL DIAGNOSTICS
Target: d_Int_Rate_ConsCred
p-value threshold: 0.05

Endogeneity (Durbin-Wu-Hausman, control-function)
H0: regressor is exogenous (p < threshold suggests endogeneity).
                variable    t_stat    p_val
  Int_Rate_ConsCred_lag1 -8.800364 0.000000
              Fin_Dostup -9.571985 0.000000
          Cred_structure  4.758207 0.000002
     Def_Zadolg_ConsCred  4.298312 0.000017
             D_top5_rozn  3.870121 0.000110
               Covid_dum -3.452796 0.000559
                Sank_dum -3.267589 0.001091
          Credit_impulse -2.883684 0.003944
             d_Mon_Shock  2.776832 0.005506
d_Inflation_Expectations -2.338539 0.019391
               d_Ex_Rate  1.464426 0.143128
   Bonds_Rate_Correct_5Y -1.352787 0.176173
               Cred_nagr  0.485984 0.626995

----------------------------------------------------------------------
MODEL: Pooled OLS
----------------------------------------------------------------------

Heteroskedasticity tests
Breusch-Pa

In [64]:
# ===== ТЕСТЫ СПЕЦИФИКАЦИИ =====

run_spec_tests(
    y, X, pooled_res, fe_res, re_res, pooled_success, fe_success, re_success
)



ТЕСТЫ СПЕЦИФИКАЦИИ

 ТЕСТ ХАУСМАНА (FE vs RE)
----------------------------------------------------------------------
H-статистика: -2.5351
p-значение: 1.000000
df: 13

Вывод: p >= 0.05 - Используйте RANDOM EFFECTS

 ТЕСТ БРЕУША-ПАГАНА (RE vs Pooled)
----------------------------------------------------------------------
LM статистика: 3069.2896
p-значение: 0.000000
N (регионов): 81, T (периодов): 77

Вывод: p < 0.05 - Есть региональные эффекты (используй RE или FE)

 F-ТЕСТ (FE vs Pooled)
----------------------------------------------------------------------
F-статистика: -72.4934
p-значение: 1.000000
df: (80, 6143)

Вывод: p >= 0.05 - Pooled адекватна


(-2.5350778437470307, 1.0, 3069.289575091302, 0.0, -72.49343671430518, 1.0)

In [65]:
###############################
# Построение линейных моделей на панельных данных (общая выборка)
# (Int_Rate_ConsCred зависимая переменная)
###############################

exog_vars_initial = [
    'Int_Rate_ConsCred_lag1',
    'Cred_nagr',
    'D_top5_rozn',
    'Fin_Dostup',
    'Credit_impulse',
    'Cred_structure',
    'Def_Zadolg_ConsCred',
    'd_Mon_Shock',
    'ln_New_Loans_ConsCred',
    'Bonds_Rate_Correct_5Y',
    'd_Ex_Rate',
    'd_Inflation_Expectations',
    'Covid_dum',
    'Sank_dum',
    'd_Mon_Shock_Covid'
]

dependent_var = 'Int_Rate_ConsCred'

y, X, pooled_res, fe_res, re_res, pooled_success, fe_success, re_success = run_panel_regressions(
    df_reg, df_reg, dependent_var, exog_vars_initial, cov_type='clustered', cluster_entity=True, df_name='df_reg'
)

# Сохраняем результаты модели для экспорта
pooled_res_2 = pooled_res if pooled_success else None
fe_res_2 = fe_res if fe_success else None
re_res_2 = re_res if re_success else None

ПАНЕЛЬНАЯ РЕГРЕССИЯ: POOL + FE + RE (ОБЩАЯ ВЫБОРКА)
Зависимая переменная: Int_Rate_ConsCred

МОДЕЛЬ 1: POOLED OLS
                          PooledOLS Estimation Summary                          
Dep. Variable:      Int_Rate_ConsCred   R-squared:                        0.9920
Estimator:                  PooledOLS   R-squared (Between):              0.9999
No. Observations:                6237   R-squared (Within):               0.9117
Date:                Tue, Dec 30 2025   R-squared (Overall):              0.9920
Time:                        15:11:57   Log-likelihood                -1.214e+04
Cov. Estimator:             Clustered                                           
                                        F-statistic:                   5.504e+04
Entities:                          81   P-value                           0.0000
Avg Obs:                       77.000   Distribution:                 F(14,6223)
Min Obs:                       77.000                                       

In [66]:
run_panel_model_diagnostics(
    y, X, pooled_res, fe_res, re_res,
    pooled_success, fe_success, re_success
)



MODEL DIAGNOSTICS
Target: Int_Rate_ConsCred
p-value threshold: 0.05

Endogeneity (Durbin-Wu-Hausman, control-function)
H0: regressor is exogenous (p < threshold suggests endogeneity).
                variable    t_stat    p_val
  Int_Rate_ConsCred_lag1 -8.699155 0.000000
              Fin_Dostup -9.537021 0.000000
          Cred_structure  4.723820 0.000002
     Def_Zadolg_ConsCred  4.243280 0.000022
             D_top5_rozn  3.818167 0.000136
               Covid_dum -3.331590 0.000869
                Sank_dum -3.233849 0.001228
          Credit_impulse -2.805231 0.005044
             d_Mon_Shock  2.533270 0.011325
d_Inflation_Expectations -2.320949 0.020322
               d_Ex_Rate  1.419096 0.155921
   Bonds_Rate_Correct_5Y -1.374440 0.169354
       d_Mon_Shock_Covid -0.585858 0.557992
               Cred_nagr  0.434150 0.664194

----------------------------------------------------------------------
MODEL: Pooled OLS
-----------------------------------------------------------------

In [67]:
# ===== ТЕСТЫ СПЕЦИФИКАЦИИ =====
print("\n" + "="*70)
print("ТЕСТЫ СПЕЦИФИКАЦИИ")
print("="*70)


run_spec_tests(
    y, X, pooled_res, fe_res, re_res, pooled_success, fe_success, re_success
)



ТЕСТЫ СПЕЦИФИКАЦИИ

ТЕСТЫ СПЕЦИФИКАЦИИ

 ТЕСТ ХАУСМАНА (FE vs RE)
----------------------------------------------------------------------
H-статистика: -5.0107
p-значение: 1.000000
df: 14

Вывод: p >= 0.05 - Используйте RANDOM EFFECTS

 ТЕСТ БРЕУША-ПАГАНА (RE vs Pooled)
----------------------------------------------------------------------
LM статистика: 3068.7544
p-значение: 0.000000
N (регионов): 81, T (периодов): 77

Вывод: p < 0.05 - Есть региональные эффекты (используй RE или FE)

 F-ТЕСТ (FE vs Pooled)
----------------------------------------------------------------------
F-статистика: -72.3630
p-значение: 1.000000
df: (80, 6142)

Вывод: p >= 0.05 - Pooled адекватна


(-5.010722357902418, 1.0, 3068.754405804813, 0.0, -72.3629668056258, 1.0)

In [68]:
###############################
# Построение линейных моделей на панельных данных (общая выборка)
# (Int_Rate_ConsCred зависимая переменная)
###############################

exog_vars_initial = [
    'Int_Rate_ConsCred_lag1',
    'Cred_nagr',
    'D_top5_rozn',
    'Fin_Dostup',
    'Credit_impulse',
    'Cred_structure',
    'Def_Zadolg_ConsCred',
    'd_Mon_Shock',
    'ln_New_Loans_ConsCred',
    'Bonds_Rate_Correct_5Y',
    'd_Ex_Rate',
    'd_Inflation_Expectations',
    'Covid_dum',
    'Sank_dum',
    'd_Mon_Shock_Cl1',
    'd_Mon_Shock_Cl2'
]

dependent_var = 'Int_Rate_ConsCred'

y, X, pooled_res, fe_res, re_res, pooled_success, fe_success, re_success = run_panel_regressions(
    df_reg, df_reg, dependent_var, exog_vars_initial, cov_type='clustered', cluster_entity=True, df_name='df_reg'
)


ПАНЕЛЬНАЯ РЕГРЕССИЯ: POOL + FE + RE (ОБЩАЯ ВЫБОРКА)
Зависимая переменная: Int_Rate_ConsCred

МОДЕЛЬ 1: POOLED OLS
                          PooledOLS Estimation Summary                          
Dep. Variable:      Int_Rate_ConsCred   R-squared:                        0.9920
Estimator:                  PooledOLS   R-squared (Between):              0.9999
No. Observations:                6237   R-squared (Within):               0.9115
Date:                Tue, Dec 30 2025   R-squared (Overall):              0.9920
Time:                        15:11:59   Log-likelihood                -1.215e+04
Cov. Estimator:             Clustered                                           
                                        F-statistic:                    5.13e+04
Entities:                          81   P-value                           0.0000
Avg Obs:                       77.000   Distribution:                 F(15,6222)
Min Obs:                       77.000                                       

In [69]:
run_panel_model_diagnostics(
    y, X, pooled_res, fe_res, re_res,
    pooled_success, fe_success, re_success
)



MODEL DIAGNOSTICS
Target: Int_Rate_ConsCred
p-value threshold: 0.05

Endogeneity (Durbin-Wu-Hausman, control-function)
H0: regressor is exogenous (p < threshold suggests endogeneity).
                variable    t_stat    p_val
  Int_Rate_ConsCred_lag1 -8.717226 0.000000
              Fin_Dostup -9.589695 0.000000
          Cred_structure  4.751587 0.000002
     Def_Zadolg_ConsCred  4.285470 0.000019
             D_top5_rozn  3.868407 0.000111
               Covid_dum -3.445779 0.000573
                Sank_dum -3.285680 0.001023
         d_Mon_Shock_Cl1  3.160143 0.001584
         d_Mon_Shock_Cl2  3.031193 0.002446
          Credit_impulse -2.866574 0.004163
d_Inflation_Expectations -2.366237 0.018000
             d_Mon_Shock  2.144821 0.032006
               d_Ex_Rate  1.491243 0.135948
   Bonds_Rate_Correct_5Y -1.305909 0.191632
               Cred_nagr  0.495605 0.620191

----------------------------------------------------------------------
MODEL: Pooled OLS
---------------------

In [70]:
# Сохраняем результаты модели для экспорта
pooled_res_3 = pooled_res if pooled_success else None
fe_res_3 = fe_res if fe_success else None
re_res_3 = re_res if re_success else None


In [71]:
# ===== ТЕСТЫ СПЕЦИФИКАЦИИ =====
print("\n" + "="*70)
print("ТЕСТЫ СПЕЦИФИКАЦИИ")
print("="*70)


run_spec_tests(
    y, X, pooled_res, fe_res, re_res, pooled_success, fe_success, re_success
)



ТЕСТЫ СПЕЦИФИКАЦИИ

ТЕСТЫ СПЕЦИФИКАЦИИ

 ТЕСТ ХАУСМАНА (FE vs RE)
----------------------------------------------------------------------
H-статистика: -2.8999
p-значение: 1.000000
df: 15

Вывод: p >= 0.05 - Используйте RANDOM EFFECTS

 ТЕСТ БРЕУША-ПАГАНА (RE vs Pooled)
----------------------------------------------------------------------
LM статистика: 3069.2886
p-значение: 0.000000
N (регионов): 81, T (периодов): 77

Вывод: p < 0.05 - Есть региональные эффекты (используй RE или FE)

 F-ТЕСТ (FE vs Pooled)
----------------------------------------------------------------------
F-статистика: -72.4717
p-значение: 1.000000
df: (80, 6141)

Вывод: p >= 0.05 - Pooled адекватна


(-2.8998918229055466, 1.0, 3069.2886077301946, 0.0, -72.4716673045289, 1.0)

In [72]:
###############################
# Построение линейных моделей на панельных данных (кластер 1)
# (Int_Rate_ConsCred зависимая переменная)
###############################

exog_vars_initial = [
    'Int_Rate_ConsCred_lag1',
    'Cred_nagr',
    'D_top5_rozn',
    'Fin_Dostup',
    'Credit_impulse',
    'Cred_structure',
    'Def_Zadolg_ConsCred',
    #'ln_New_Loans_Progr',
    'd_Mon_Shock',
    'ln_New_Loans_ConsCred',
    'Bonds_Rate_Correct_5Y',
    'd_Ex_Rate',
    'd_Inflation_Expectations'                          
]

dependent_var = 'Int_Rate_ConsCred'

y, X, pooled_res, fe_res, re_res, pooled_success, fe_success, re_success = run_panel_regressions(
    df_reg_clus_one, df_reg, dependent_var, exog_vars_initial, cov_type='clustered', cluster_entity=True, df_name='df_reg_clus_one'
)


ПАНЕЛЬНАЯ РЕГРЕССИЯ: POOL + FE + RE (КЛАСТЕР 1)
Зависимая переменная: Int_Rate_ConsCred

МОДЕЛЬ 1: POOLED OLS
                          PooledOLS Estimation Summary                          
Dep. Variable:      Int_Rate_ConsCred   R-squared:                        0.9892
Estimator:                  PooledOLS   R-squared (Between):              0.9997
No. Observations:                4312   R-squared (Within):               0.8796
Date:                Tue, Dec 30 2025   R-squared (Overall):              0.9892
Time:                        15:12:00   Log-likelihood                   -9056.0
Cov. Estimator:             Clustered                                           
                                        F-statistic:                    3.59e+04
Entities:                          56   P-value                           0.0000
Avg Obs:                       77.000   Distribution:                 F(11,4301)
Min Obs:                       77.000                                           

In [73]:
run_panel_model_diagnostics(
    y, X, pooled_res, fe_res, re_res,
    pooled_success, fe_success, re_success
)



MODEL DIAGNOSTICS
Target: Int_Rate_ConsCred
p-value threshold: 0.05

Endogeneity (Durbin-Wu-Hausman, control-function)
H0: regressor is exogenous (p < threshold suggests endogeneity).
                variable     t_stat    p_val
  Int_Rate_ConsCred_lag1 -13.399583 0.000000
             D_top5_rozn   9.343608 0.000000
              Fin_Dostup   9.170240 0.000000
          Credit_impulse  -8.832353 0.000000
          Cred_structure  13.070755 0.000000
     Def_Zadolg_ConsCred  10.565159 0.000000
             d_Mon_Shock   8.699581 0.000000
d_Inflation_Expectations  -4.555399 0.000005
               Cred_nagr   2.747775 0.006025
               d_Ex_Rate   2.403684 0.016273
   Bonds_Rate_Correct_5Y  -0.912811 0.361393

----------------------------------------------------------------------
MODEL: Pooled OLS
----------------------------------------------------------------------

Heteroskedasticity tests
Breusch-Pagan: stat=134.0367, p=0.000000
White:         stat=1254.8891, p=0.000000

Norm

In [74]:
# Сохраняем результаты модели для экспорта
pooled_res_4 = pooled_res if pooled_success else None
fe_res_4 = fe_res if fe_success else None
re_res_4 = re_res if re_success else None


In [75]:
# ===== ТЕСТЫ СПЕЦИФИКАЦИИ =====
print("\n" + "="*70)
print("ТЕСТЫ СПЕЦИФИКАЦИИ")
print("="*70)


run_spec_tests(
    y, X, pooled_res, fe_res, re_res, pooled_success, fe_success, re_success
)



ТЕСТЫ СПЕЦИФИКАЦИИ

ТЕСТЫ СПЕЦИФИКАЦИИ

 ТЕСТ ХАУСМАНА (FE vs RE)
----------------------------------------------------------------------
H-статистика: 0.4898
p-значение: 0.999999
df: 11

Вывод: p >= 0.05 - Используйте RANDOM EFFECTS

 ТЕСТ БРЕУША-ПАГАНА (RE vs Pooled)
----------------------------------------------------------------------
LM статистика: 2078.2224
p-значение: 0.000000
N (регионов): 56, T (периодов): 77

Вывод: p < 0.05 - Есть региональные эффекты (используй RE или FE)

 F-ТЕСТ (FE vs Pooled)
----------------------------------------------------------------------
F-статистика: -76.3342
p-значение: 1.000000
df: (55, 4245)

Вывод: p >= 0.05 - Pooled адекватна


(0.4897702094357044,
 0.9999987688463157,
 2078.2224173032946,
 0.0,
 -76.33418840265337,
 1.0)

In [76]:
###############################
# Построение линейных моделей на панельных данных (кластер 2)
# (Int_Rate_ConsCred зависимая переменная)
###############################

exog_vars_initial = [
    'Int_Rate_ConsCred_lag1',
    'Cred_nagr',
    'D_top5_rozn',
    'Fin_Dostup',
    'Credit_impulse',
    'Cred_structure',
    'Def_Zadolg_ConsCred',
    #'ln_New_Loans_Progr',
    'd_Mon_Shock',
    'ln_New_Loans_ConsCred',
    'Bonds_Rate_Correct_5Y',
    'd_Ex_Rate',
    'd_Inflation_Expectations'                          
]

dependent_var = 'Int_Rate_ConsCred'

y, X, pooled_res, fe_res, re_res, pooled_success, fe_success, re_success = run_panel_regressions(
    df_reg_clus_two, df_reg, dependent_var, exog_vars_initial, cov_type='clustered', cluster_entity=True, df_name='df_reg_clus_two'
)


ПАНЕЛЬНАЯ РЕГРЕССИЯ: POOL + FE + RE (КЛАСТЕР 2)
Зависимая переменная: Int_Rate_ConsCred

МОДЕЛЬ 1: POOLED OLS
                          PooledOLS Estimation Summary                          
Dep. Variable:      Int_Rate_ConsCred   R-squared:                        0.9928
Estimator:                  PooledOLS   R-squared (Between):              0.9998
No. Observations:                 462   R-squared (Within):               0.9403
Date:                Tue, Dec 30 2025   R-squared (Overall):              0.9928
Time:                        15:12:02   Log-likelihood                   -893.73
Cov. Estimator:             Clustered                                           
                                        F-statistic:                      5665.4
Entities:                           6   P-value                           0.0000
Avg Obs:                       77.000   Distribution:                  F(11,451)
Min Obs:                       77.000                                           

In [77]:
run_panel_model_diagnostics(
    y, X, pooled_res, fe_res, re_res,
    pooled_success, fe_success, re_success
)



MODEL DIAGNOSTICS
Target: Int_Rate_ConsCred
p-value threshold: 0.05

Endogeneity (Durbin-Wu-Hausman, control-function)
H0: regressor is exogenous (p < threshold suggests endogeneity).
                variable    t_stat    p_val
  Int_Rate_ConsCred_lag1 -3.735204 0.000212
d_Inflation_Expectations  3.265076 0.001178
   Bonds_Rate_Correct_5Y -3.142609 0.001785
               d_Ex_Rate -3.059526 0.002349
               Cred_nagr -2.961683 0.003222
          Cred_structure  2.734039 0.006503
             d_Mon_Shock  2.608438 0.009398
             D_top5_rozn -1.454618 0.146472
              Fin_Dostup  1.252719 0.210958
     Def_Zadolg_ConsCred -0.509774 0.610460
          Credit_impulse -0.433342 0.664974

----------------------------------------------------------------------
MODEL: Pooled OLS
----------------------------------------------------------------------

Heteroskedasticity tests
Breusch-Pagan: stat=79.5277, p=0.000000
White:         stat=295.5121, p=0.000000

Normality tests
Ja

In [78]:
# Сохраняем результаты модели для экспорта
pooled_res_5 = pooled_res if pooled_success else None
fe_res_5 = fe_res if fe_success else None
re_res_5 = re_res if re_success else None


In [79]:
# ===== ТЕСТЫ СПЕЦИФИКАЦИИ =====
print("\n" + "="*70)
print("ТЕСТЫ СПЕЦИФИКАЦИИ")
print("="*70)


run_spec_tests(
    y, X, pooled_res, fe_res, re_res, pooled_success, fe_success, re_success
)



ТЕСТЫ СПЕЦИФИКАЦИИ

ТЕСТЫ СПЕЦИФИКАЦИИ

 ТЕСТ ХАУСМАНА (FE vs RE)
----------------------------------------------------------------------
H-статистика: 21.9805
p-значение: 0.024525
df: 11

Вывод: p < 0.05 - Используйте FIXED EFFECTS

 ТЕСТ БРЕУША-ПАГАНА (RE vs Pooled)
----------------------------------------------------------------------
LM статистика: 224.0994
p-значение: 0.000000
N (регионов): 6, T (периодов): 77

Вывод: p < 0.05 - Есть региональные эффекты (используй RE или FE)

 F-ТЕСТ (FE vs Pooled)
----------------------------------------------------------------------
F-статистика: -88.1347
p-значение: 1.000000
df: (5, 445)

Вывод: p >= 0.05 - Pooled адекватна


(21.980519291315495,
 0.024524598458227076,
 224.09940360774905,
 0.0,
 -88.1346723192737,
 1.0)

In [80]:
###############################
# Построение линейных моделей на панельных данных (кластер 3)
# (Int_Rate_ConsCred зависимая переменная)
###############################

exog_vars_initial = [
    'Int_Rate_ConsCred_lag1',
    'Cred_nagr',
    'D_top5_rozn',
    'Fin_Dostup',
    'Credit_impulse',
    'Cred_structure',
    'Def_Zadolg_ConsCred',
    #'ln_New_Loans_Progr',
    'd_Mon_Shock',
    'ln_New_Loans_ConsCred',
    'Bonds_Rate_Correct_5Y',
    'd_Ex_Rate',
    'd_Inflation_Expectations'                             
]

dependent_var = 'Int_Rate_ConsCred'

y, X, pooled_res, fe_res, re_res, pooled_success, fe_success, re_success = run_panel_regressions(
    df_reg_clus_three, df_reg, dependent_var, exog_vars_initial, cov_type='clustered', cluster_entity=True, df_name='df_reg_clus_three'
)


ПАНЕЛЬНАЯ РЕГРЕССИЯ: POOL + FE + RE (КЛАСТЕР 3)
Зависимая переменная: Int_Rate_ConsCred

МОДЕЛЬ 1: POOLED OLS
                          PooledOLS Estimation Summary                          
Dep. Variable:      Int_Rate_ConsCred   R-squared:                        0.9977
Estimator:                  PooledOLS   R-squared (Between):              0.9999
No. Observations:                1463   R-squared (Within):               0.9742
Date:                Tue, Dec 30 2025   R-squared (Overall):              0.9977
Time:                        15:12:02   Log-likelihood                   -1893.0
Cov. Estimator:             Clustered                                           
                                        F-statistic:                   5.733e+04
Entities:                          19   P-value                           0.0000
Avg Obs:                       77.000   Distribution:                 F(11,1452)
Min Obs:                       77.000                                           

In [81]:
run_panel_model_diagnostics(
    y, X, pooled_res, fe_res, re_res,
    pooled_success, fe_success, re_success
)



MODEL DIAGNOSTICS
Target: Int_Rate_ConsCred
p-value threshold: 0.05

Endogeneity (Durbin-Wu-Hausman, control-function)
H0: regressor is exogenous (p < threshold suggests endogeneity).
                variable    t_stat    p_val
  Int_Rate_ConsCred_lag1 -6.746929 0.000000
          Credit_impulse -5.934013 0.000000
             D_top5_rozn  5.733005 0.000000
               Cred_nagr  4.431746 0.000010
             d_Mon_Shock  3.909068 0.000097
   Bonds_Rate_Correct_5Y  3.631900 0.000291
d_Inflation_Expectations -2.879198 0.004045
               d_Ex_Rate  2.118179 0.034330
     Def_Zadolg_ConsCred  1.396686 0.162722
              Fin_Dostup -0.985051 0.324763
          Cred_structure -0.194788 0.845587

----------------------------------------------------------------------
MODEL: Pooled OLS
----------------------------------------------------------------------

Heteroskedasticity tests
Breusch-Pagan: stat=186.5211, p=0.000000
White:         stat=627.9528, p=0.000000

Normality tests
J

In [82]:
# Сохраняем результаты модели для экспорта
pooled_res_6 = pooled_res if pooled_success else None
fe_res_6 = fe_res if fe_success else None
re_res_6 = re_res if re_success else None


In [83]:
# ===== ТЕСТЫ СПЕЦИФИКАЦИИ =====
print("\n" + "="*70)
print("ТЕСТЫ СПЕЦИФИКАЦИИ")
print("="*70)


run_spec_tests(
    y, X, pooled_res, fe_res, re_res, pooled_success, fe_success, re_success
)



ТЕСТЫ СПЕЦИФИКАЦИИ

ТЕСТЫ СПЕЦИФИКАЦИИ

 ТЕСТ ХАУСМАНА (FE vs RE)
----------------------------------------------------------------------
H-статистика: -16.4403
p-значение: 1.000000
df: 11

Вывод: p >= 0.05 - Используйте RANDOM EFFECTS

 ТЕСТ БРЕУША-ПАГАНА (RE vs Pooled)
----------------------------------------------------------------------
LM статистика: 688.8280
p-значение: 0.000000
N (регионов): 19, T (периодов): 77

Вывод: p < 0.05 - Есть региональные эффекты (используй RE или FE)

 F-ТЕСТ (FE vs Pooled)
----------------------------------------------------------------------
F-статистика: -67.1276
p-значение: 1.000000
df: (18, 1433)

Вывод: p >= 0.05 - Pooled адекватна


(-16.44029170348216, 1.0, 688.8280092938837, 0.0, -67.1275847045801, 1.0)

In [84]:
###############################
# Построение линейных моделей на панельных данных (общая выборка) ROISFIX
# (Int_Rate_ConsCred зависимая переменная)
###############################

exog_vars_initial = [
    'Int_Rate_ConsCred_lag1',
    'Cred_nagr',
    'D_top5_rozn',
    'Fin_Dostup',
    'Credit_impulse',
    'Cred_structure',
    'Def_Zadolg_ConsCred',
    #'ln_New_Loans_Progr',
    'ROISFIX',
    'ln_New_Loans_ConsCred',
    'Bonds_Rate_Correct_5Y',
    'd_Ex_Rate',
    'd_Inflation_Expectations'                          
]

dependent_var = 'Int_Rate_ConsCred'

y, X, pooled_res, fe_res, re_res, pooled_success, fe_success, re_success = run_panel_regressions(
    df_reg, df_reg, dependent_var, exog_vars_initial, cov_type='clustered', cluster_entity=True, df_name='df_reg'
)


ПАНЕЛЬНАЯ РЕГРЕССИЯ: POOL + FE + RE (ОБЩАЯ ВЫБОРКА) - ROISFIX
Зависимая переменная: Int_Rate_ConsCred

МОДЕЛЬ 1: POOLED OLS
                          PooledOLS Estimation Summary                          
Dep. Variable:      Int_Rate_ConsCred   R-squared:                        0.9930
Estimator:                  PooledOLS   R-squared (Between):              0.9999
No. Observations:                6317   R-squared (Within):               0.9225
Date:                Tue, Dec 30 2025   R-squared (Overall):              0.9930
Time:                        15:12:03   Log-likelihood                -1.187e+04
Cov. Estimator:             Clustered                                           
                                        F-statistic:                   8.117e+04
Entities:                          81   P-value                           0.0000
Avg Obs:                       77.988   Distribution:                 F(11,6306)
Min Obs:                       77.000                             

In [85]:
run_panel_model_diagnostics(
    y, X, pooled_res, fe_res, re_res,
    pooled_success, fe_success, re_success
)



MODEL DIAGNOSTICS
Target: Int_Rate_ConsCred
p-value threshold: 0.05

Endogeneity (Durbin-Wu-Hausman, control-function)
H0: regressor is exogenous (p < threshold suggests endogeneity).
                variable     t_stat    p_val
              Fin_Dostup -11.826879 0.000000
  Int_Rate_ConsCred_lag1  -6.189856 0.000000
   Bonds_Rate_Correct_5Y  -3.480997 0.000503
             D_top5_rozn   2.584872 0.009764
     Def_Zadolg_ConsCred   2.352010 0.018703
          Cred_structure   1.968086 0.049102
                 ROISFIX  -1.869859 0.061550
          Credit_impulse  -1.351591 0.176555
               d_Ex_Rate  -0.504757 0.613747
               Cred_nagr  -0.186881 0.851760
d_Inflation_Expectations   0.145239 0.884527

----------------------------------------------------------------------
MODEL: Pooled OLS
----------------------------------------------------------------------

Heteroskedasticity tests
Breusch-Pagan: stat=93.8975, p=0.000000
White:         stat=1080.8861, p=0.000000

Norma

In [86]:
# Сохраняем результаты модели для экспорта
pooled_res_7 = pooled_res if pooled_success else None
fe_res_7 = fe_res if fe_success else None
re_res_7 = re_res if re_success else None


In [87]:
# ===== ТЕСТЫ СПЕЦИФИКАЦИИ =====
from scipy.stats import f

print("\n" + "="*70)
print("ТЕСТЫ СПЕЦИФИКАЦИИ")
print("="*70)


run_spec_tests(
    y, X, pooled_res, fe_res, re_res, pooled_success, fe_success, re_success
)



ТЕСТЫ СПЕЦИФИКАЦИИ

ТЕСТЫ СПЕЦИФИКАЦИИ

 ТЕСТ ХАУСМАНА (FE vs RE)
----------------------------------------------------------------------
H-статистика: 5.3114
p-значение: 0.915151
df: 11

Вывод: p >= 0.05 - Используйте RANDOM EFFECTS

 ТЕСТ БРЕУША-ПАГАНА (RE vs Pooled)
----------------------------------------------------------------------
LM статистика: 3050.5367
p-значение: 0.000000
N (регионов): 81, T (периодов): 77

Вывод: p < 0.05 - Есть региональные эффекты (используй RE или FE)

 F-ТЕСТ (FE vs Pooled)
----------------------------------------------------------------------
F-статистика: -62.6207
p-значение: 1.000000
df: (80, 6226)

Вывод: p >= 0.05 - Pooled адекватна


(5.311438719082098,
 0.9151507893753551,
 3050.53671159259,
 0.0,
 -62.62069362726531,
 1.0)

In [88]:
###############################
# Построение линейных моделей на панельных данных (кластер 1 ROISFIX)
# (Int_Rate_ConsCred зависимая переменная)
###############################

exog_vars_initial = [
    'Int_Rate_ConsCred_lag1',
    'Cred_nagr',
    'D_top5_rozn',
    'Fin_Dostup',
    'Credit_impulse',
    'Cred_structure',
    'Def_Zadolg_ConsCred',
    #'ln_New_Loans_Progr',
    'ROISFIX',
    'ln_New_Loans_ConsCred',
    'Bonds_Rate_Correct_5Y',
    'd_Ex_Rate',
    'd_Inflation_Expectations'                          
]

dependent_var = 'Int_Rate_ConsCred'

y, X, pooled_res, fe_res, re_res, pooled_success, fe_success, re_success = run_panel_regressions(
    df_reg_clus_one, df_reg, dependent_var, exog_vars_initial, cov_type='clustered', cluster_entity=True, df_name='df_reg_clus_one'
)


ПАНЕЛЬНАЯ РЕГРЕССИЯ: POOL + FE + RE (КЛАСТЕР 1 ROISFIX)
Зависимая переменная: Int_Rate_ConsCred

МОДЕЛЬ 1: POOLED OLS
                          PooledOLS Estimation Summary                          
Dep. Variable:      Int_Rate_ConsCred   R-squared:                        0.9915
Estimator:                  PooledOLS   R-squared (Between):              0.9998
No. Observations:                4367   R-squared (Within):               0.9035
Date:                Tue, Dec 30 2025   R-squared (Overall):              0.9915
Time:                        15:12:05   Log-likelihood                   -8654.3
Cov. Estimator:             Clustered                                           
                                        F-statistic:                   4.598e+04
Entities:                          56   P-value                           0.0000
Avg Obs:                       77.982   Distribution:                 F(11,4356)
Min Obs:                       77.000                                   

In [89]:
run_panel_model_diagnostics(
    y, X, pooled_res, fe_res, re_res,
    pooled_success, fe_success, re_success
)



MODEL DIAGNOSTICS
Target: Int_Rate_ConsCred
p-value threshold: 0.05

Endogeneity (Durbin-Wu-Hausman, control-function)
H0: regressor is exogenous (p < threshold suggests endogeneity).
                variable    t_stat    p_val
  Int_Rate_ConsCred_lag1 -5.628460 0.000000
   Bonds_Rate_Correct_5Y -4.970020 0.000001
          Cred_structure  3.964806 0.000075
                 ROISFIX -3.526505 0.000425
     Def_Zadolg_ConsCred  3.297894 0.000982
              Fin_Dostup  3.279590 0.001048
             D_top5_rozn  3.259814 0.001123
          Credit_impulse -2.168490 0.030175
               d_Ex_Rate  0.293797 0.768927
d_Inflation_Expectations -0.202745 0.839344
               Cred_nagr -0.068565 0.945339

----------------------------------------------------------------------
MODEL: Pooled OLS
----------------------------------------------------------------------

Heteroskedasticity tests
Breusch-Pagan: stat=76.3436, p=0.000000
White:         stat=705.5709, p=0.000000

Normality tests
Ja

In [90]:
# Сохраняем результаты модели для экспорта
pooled_res_8 = pooled_res if pooled_success else None
fe_res_8 = fe_res if fe_success else None
re_res_8 = re_res if re_success else None


In [91]:
# ===== ТЕСТЫ СПЕЦИФИКАЦИИ =====
print("\n" + "="*70)
print("ТЕСТЫ СПЕЦИФИКАЦИИ")
print("="*70)


run_spec_tests(
    y, X, pooled_res, fe_res, re_res, pooled_success, fe_success, re_success
)



ТЕСТЫ СПЕЦИФИКАЦИИ

ТЕСТЫ СПЕЦИФИКАЦИИ

 ТЕСТ ХАУСМАНА (FE vs RE)
----------------------------------------------------------------------
H-статистика: 11.4670
p-значение: 0.405008
df: 11

Вывод: p >= 0.05 - Используйте RANDOM EFFECTS

 ТЕСТ БРЕУША-ПАГАНА (RE vs Pooled)
----------------------------------------------------------------------
LM статистика: 2104.2002
p-значение: 0.000000
N (регионов): 56, T (периодов): 77

Вывод: p < 0.05 - Есть региональные эффекты (используй RE или FE)

 F-ТЕСТ (FE vs Pooled)
----------------------------------------------------------------------
F-статистика: -67.2494
p-значение: 1.000000
df: (55, 4301)

Вывод: p >= 0.05 - Pooled адекватна


(11.467009660672302,
 0.4050075465499736,
 2104.2002117790976,
 0.0,
 -67.24940115636679,
 1.0)

In [92]:
###############################
# Построение линейных моделей на панельных данных (кластер 2 ROISFIX)
# (Int_Rate_ConsCred зависимая переменная)
###############################

exog_vars_initial = [
    'Int_Rate_ConsCred_lag1',
    'Cred_nagr',
    'D_top5_rozn',
    'Fin_Dostup',
    'Credit_impulse',
    'Cred_structure',
    'Def_Zadolg_ConsCred',
    #'ln_New_Loans_Progr',
    'ROISFIX',
    'ln_New_Loans_ConsCred',
    'Bonds_Rate_Correct_5Y',
    'd_Ex_Rate',
    'd_Inflation_Expectations'                          
]

dependent_var = 'Int_Rate_ConsCred'

y, X, pooled_res, fe_res, re_res, pooled_success, fe_success, re_success = run_panel_regressions(
    df_reg_clus_two, df_reg, dependent_var, exog_vars_initial, cov_type='clustered', cluster_entity=True, df_name='df_reg_clus_two'
)


ПАНЕЛЬНАЯ РЕГРЕССИЯ: POOL + FE + RE (КЛАСТЕР 2 ROISFIX)
Зависимая переменная: Int_Rate_ConsCred

МОДЕЛЬ 1: POOLED OLS
                          PooledOLS Estimation Summary                          
Dep. Variable:      Int_Rate_ConsCred   R-squared:                        0.9937
Estimator:                  PooledOLS   R-squared (Between):              0.9999
No. Observations:                 468   R-squared (Within):               0.9470
Date:                Tue, Dec 30 2025   R-squared (Overall):              0.9937
Time:                        15:12:07   Log-likelihood                   -874.44
Cov. Estimator:             Clustered                                           
                                        F-statistic:                      6519.2
Entities:                           6   P-value                           0.0000
Avg Obs:                       78.000   Distribution:                  F(11,457)
Min Obs:                       78.000                                   

In [93]:
run_panel_model_diagnostics(
    y, X, pooled_res, fe_res, re_res,
    pooled_success, fe_success, re_success
)



MODEL DIAGNOSTICS
Target: Int_Rate_ConsCred
p-value threshold: 0.05

Endogeneity (Durbin-Wu-Hausman, control-function)
H0: regressor is exogenous (p < threshold suggests endogeneity).
                variable    t_stat    p_val
  Int_Rate_ConsCred_lag1 -4.333727 0.000018
   Bonds_Rate_Correct_5Y -4.159155 0.000038
d_Inflation_Expectations  4.145732 0.000040
               d_Ex_Rate -4.063492 0.000057
               Cred_nagr -3.852940 0.000133
          Cred_structure  2.074996 0.038547
                 ROISFIX  1.537468 0.124872
             D_top5_rozn -1.264619 0.206654
          Credit_impulse  1.146487 0.252195
              Fin_Dostup  0.571597 0.567876
     Def_Zadolg_ConsCred  0.042824 0.965861

----------------------------------------------------------------------
MODEL: Pooled OLS
----------------------------------------------------------------------

Heteroskedasticity tests
Breusch-Pagan: stat=83.2954, p=0.000000
White:         stat=266.9677, p=0.000000

Normality tests
Ja

In [94]:
# Сохраняем результаты модели для экспорта
pooled_res_9 = pooled_res if pooled_success else None
fe_res_9 = fe_res if fe_success else None
re_res_9 = re_res if re_success else None


In [95]:
# ===== ТЕСТЫ СПЕЦИФИКАЦИИ =====
print("\n" + "="*70)
print("ТЕСТЫ СПЕЦИФИКАЦИИ")
print("="*70)


run_spec_tests(
    y, X, pooled_res, fe_res, re_res, pooled_success, fe_success, re_success
)



ТЕСТЫ СПЕЦИФИКАЦИИ

ТЕСТЫ СПЕЦИФИКАЦИИ

 ТЕСТ ХАУСМАНА (FE vs RE)
----------------------------------------------------------------------
H-статистика: -1.8520
p-значение: 1.000000
df: 11

Вывод: p >= 0.05 - Используйте RANDOM EFFECTS

 ТЕСТ БРЕУША-ПАГАНА (RE vs Pooled)
----------------------------------------------------------------------
LM статистика: 228.3614
p-значение: 0.000000
N (регионов): 6, T (периодов): 78

Вывод: p < 0.05 - Есть региональные эффекты (используй RE или FE)

 F-ТЕСТ (FE vs Pooled)
----------------------------------------------------------------------
F-статистика: -88.3895
p-значение: 1.000000
df: (5, 451)

Вывод: p >= 0.05 - Pooled адекватна


(-1.8519949689107973, 1.0, 228.36136996895334, 0.0, -88.38945308116432, 1.0)

In [96]:
###############################
# Построение линейных моделей на панельных данных (кластер 3 ROISFIX)
# (Int_Rate_ConsCred зависимая переменная)
###############################

exog_vars_initial = [
    'Int_Rate_ConsCred_lag1',
    'Cred_nagr',
    'D_top5_rozn',
    'Fin_Dostup',
    'Credit_impulse',
    'Cred_structure',
    'Def_Zadolg_ConsCred',
    #'ln_New_Loans_Progr',
    'ROISFIX',
    'ln_New_Loans_ConsCred',
    'Bonds_Rate_Correct_5Y',
    'd_Ex_Rate',
    'd_Inflation_Expectations'                             
]

dependent_var = 'Int_Rate_ConsCred'

y, X, pooled_res, fe_res, re_res, pooled_success, fe_success, re_success = run_panel_regressions(
    df_reg_clus_three, df_reg, dependent_var, exog_vars_initial, cov_type='clustered', cluster_entity=True, df_name='df_reg_clus_three'
)


ПАНЕЛЬНАЯ РЕГРЕССИЯ: POOL + FE + RE (КЛАСТЕР 3 ROISFIX)
Зависимая переменная: Int_Rate_ConsCred

МОДЕЛЬ 1: POOLED OLS
                          PooledOLS Estimation Summary                          
Dep. Variable:      Int_Rate_ConsCred   R-squared:                        0.9984
Estimator:                  PooledOLS   R-squared (Between):              1.0000
No. Observations:                1482   R-squared (Within):               0.9819
Date:                Tue, Dec 30 2025   R-squared (Overall):              0.9984
Time:                        15:12:08   Log-likelihood                   -1633.2
Cov. Estimator:             Clustered                                           
                                        F-statistic:                   8.493e+04
Entities:                          19   P-value                           0.0000
Avg Obs:                       78.000   Distribution:                 F(11,1471)
Min Obs:                       78.000                                   

In [97]:
run_panel_model_diagnostics(
    y, X, pooled_res, fe_res, re_res,
    pooled_success, fe_success, re_success
)



MODEL DIAGNOSTICS
Target: Int_Rate_ConsCred
p-value threshold: 0.05

Endogeneity (Durbin-Wu-Hausman, control-function)
H0: regressor is exogenous (p < threshold suggests endogeneity).
                variable    t_stat    p_val
              Fin_Dostup -5.108536 0.000000
     Def_Zadolg_ConsCred -4.445255 0.000009
          Cred_structure -4.333624 0.000016
               d_Ex_Rate -3.016405 0.002602
   Bonds_Rate_Correct_5Y  2.466628 0.013753
d_Inflation_Expectations  2.005562 0.045086
  Int_Rate_ConsCred_lag1 -1.910553 0.056256
               Cred_nagr -1.698005 0.089718
                 ROISFIX  1.512448 0.130635
          Credit_impulse  1.301206 0.193392
             D_top5_rozn  1.087382 0.277046

----------------------------------------------------------------------
MODEL: Pooled OLS
----------------------------------------------------------------------

Heteroskedasticity tests
Breusch-Pagan: stat=140.0152, p=0.000000
White:         stat=574.2154, p=0.000000

Normality tests
J

In [98]:
# Сохраняем результаты модели для экспорта
pooled_res_10 = pooled_res if pooled_success else None
fe_res_10 = fe_res if fe_success else None
re_res_10 = re_res if re_success else None


In [99]:
# ===== ТЕСТЫ СПЕЦИФИКАЦИИ =====
print("\n" + "="*70)
print("ТЕСТЫ СПЕЦИФИКАЦИИ")
print("="*70)


run_spec_tests(
    y, X, pooled_res, fe_res, re_res, pooled_success, fe_success, re_success
)



ТЕСТЫ СПЕЦИФИКАЦИИ

ТЕСТЫ СПЕЦИФИКАЦИИ

 ТЕСТ ХАУСМАНА (FE vs RE)
----------------------------------------------------------------------
H-статистика: 5.6793
p-значение: 0.893882
df: 11

Вывод: p >= 0.05 - Используйте RANDOM EFFECTS

 ТЕСТ БРЕУША-ПАГАНА (RE vs Pooled)
----------------------------------------------------------------------
LM статистика: 726.8895
p-значение: 0.000000
N (регионов): 19, T (периодов): 78

Вывод: p < 0.05 - Есть региональные эффекты (используй RE или FE)

 F-ТЕСТ (FE vs Pooled)
----------------------------------------------------------------------
F-статистика: -79.9841
p-значение: 1.000000
df: (18, 1452)

Вывод: p >= 0.05 - Pooled адекватна


(5.6793044211006745,
 0.8938824419305631,
 726.8895099696001,
 0.0,
 -79.9841333810772,
 1.0)

### Вывод результатов

In [100]:
from model_results_export import ModelResultsAggregator, ensure_results_dir, add_model_set, build_and_export
import os

dep_var_name = 'Int_Rate_ConsCred'
base_name = dep_var_name
if base_name.startswith(''):
    base_name = base_name[2:]
if base_name.endswith(''):
    base_name = base_name[:-4]

results_dir = ensure_results_dir('Results')

model_specs_all = [
    {
        'spec_name': 'Модель_1',
        'dependent_var': dep_var_name,
        'subsample': 'Общая выборка',
        'results': {
            'pooled': pooled_res_1,
            'fe': fe_res_1,
            're': re_res_1
        }
    },
    {
        'spec_name': 'Модель_2',
        'dependent_var': dep_var_name,
        'subsample': 'Общая выборка',
        'results': {
            'pooled': pooled_res_2,
            'fe': fe_res_2,
            're': re_res_2
        }
    },
    {
        'spec_name': 'Модель_3',
        'dependent_var': dep_var_name,
        'subsample': 'Общая выборка',
        'results': {
            'pooled': pooled_res_3,
            'fe': fe_res_3,
            're': re_res_3
        }
    },
    {
        'spec_name': 'Модель_4',
        'dependent_var': dep_var_name,
        'subsample': 'Общая выборка',
        'results': {
            'pooled': pooled_res_7,
            'fe': fe_res_7,
            're': re_res_7
        }
    },
    {
        'spec_name': 'Модель_5 (попытка)',
        'dependent_var': dep_var_name,
        'subsample': 'Общая выборка',
        'results': {
            'pooled': pooled_res_77,
            'fe': fe_res_77,
            're': re_res_77
        }
    }
]

model_specs_cluster = [
    {
        'spec_name': 'Модель_1',
        'dependent_var': dep_var_name,
        'subsample': 'Кластер 1',
        'results': {
            'pooled': pooled_res_4,
            'fe': fe_res_4,
            're': re_res_4
        }
    },
    {
        'spec_name': 'Модель_2',
        'dependent_var': dep_var_name,
        'subsample': 'Кластер 1',
        'results': {
            'pooled': pooled_res_8,
            'fe': fe_res_8,
            're': re_res_8
        }
    },
    {
        'spec_name': 'Модель_3',
        'dependent_var': dep_var_name,
        'subsample': 'Кластер 2',
        'results': {
            'pooled': pooled_res_5,
            'fe': fe_res_5,
            're': re_res_5
        }
    },
    {
        'spec_name': 'Модель_4',
        'dependent_var': dep_var_name,
        'subsample': 'Кластер 2',
        'results': {
            'pooled': pooled_res_9,
            'fe': fe_res_9,
            're': re_res_9
        }
    },
    {
        'spec_name': 'Модель_5',
        'dependent_var': dep_var_name,
        'subsample': 'Кластер 3',
        'results': {
            'pooled': pooled_res_6,
            'fe': fe_res_6,
            're': re_res_6
        }
    },
    {
        'spec_name': 'Модель_6',
        'dependent_var': dep_var_name,
        'subsample': 'Кластер 3',
        'results': {
            'pooled': pooled_res_10,
            'fe': fe_res_10,
            're': re_res_10
        }
    }
]

aggregator_all = ModelResultsAggregator()
for spec in model_specs_all:
    add_model_set(aggregator_all, spec)

out_all = os.path.join(results_dir, f"{base_name}_all_data.xlsx")
build_and_export(aggregator_all, out_all, include_pvalues=True, decimals=3)

aggregator_cluster = ModelResultsAggregator()
for spec in model_specs_cluster:
    add_model_set(aggregator_cluster, spec)

out_cluster = os.path.join(results_dir, f"{base_name}_cluster_data.xlsx")
build_and_export(aggregator_cluster, out_cluster, include_pvalues=True, decimals=3)


,Модель_1 (POOL),Модель_1 (FE),Модель_1 (RE),Модель_2 (POOL),Модель_2 (FE),Модель_2 (RE),Модель_3 (POOL),Модель_3 (FE),Модель_3 (RE),Модель_4 (POOL),Модель_4 (FE),Модель_4 (RE),Модель_5 (POOL),Модель_5 (FE),Модель_5 (RE),Модель_6 (POOL),Модель_6 (FE),Модель_6 (RE)
Зависимая переменная,Int_Rate_ConsCred,Int_Rate_ConsCred,Int_Rate_ConsCred,Int_Rate_ConsCred,Int_Rate_ConsCred,Int_Rate_ConsCred,Int_Rate_ConsCred,Int_Rate_ConsCred,Int_Rate_ConsCred,Int_Rate_ConsCred,Int_Rate_ConsCred,Int_Rate_ConsCred,Int_Rate_ConsCred,Int_Rate_ConsCred,Int_Rate_ConsCred,Int_Rate_ConsCred,Int_Rate_ConsCred,Int_Rate_ConsCred
Int_Rate_ConsCred_lag1,0.771*** (0.000),0.599*** (0.000),0.771*** (0.000),0.558*** (0.000),0.471*** (0.000),0.558*** (0.000),0.811*** (0.000),0.727*** (0.000),0.811*** (0.000),0.672*** (0.000),0.638*** (0.000),0.672*** (0.000),0.876*** (0.000),0.797*** (0.000),0.876*** (0.000),0.724*** (0.000),0.670*** (0.000),0.724*** (0.000)
Cred_nagr,-1.335 (0.385),-7.591** (0.038),-1.335 (0.385),-1.543 (0.193),-11.103*** (0.002),-1.543 (0.193),-6.899 (0.144),-6.622 (0.429),-6.899 (0.144),-10.689** (0.028),-15.189*** (0.008),-10.689** (0.028),1.024** (0.016),-3.317*** (0.000),1.024** (0.016),0.407 (0.263),-6.600*** (0.000),0.407 (0.263)
D_top5_rozn,2.112*** (0.001),21.201** (0.017),2.112*** (0.001),3.382*** (0.000),7.607 (0.105),3.382*** (0.000),-1.721 (0.148),-20.334* (0.088),-1.721 (0.148),2.911 (0.188),-10.238 (0.363),2.911 (0.188),3.795*** (0.000),1.783 (0.616),3.795*** (0.000),2.428*** (0.000),-5.111** (0.046),2.428*** (0.000)
Fin_Dostup,-0.025*** (0.000),0.049 (0.433),-0.025*** (0.000),0.006 (0.372),0.089 (0.126),0.006 (0.372),0.017 (0.195),-0.088 (0.176),0.017 (0.195),-0.009 (0.593),-0.024 (0.544),-0.009 (0.593),-0.062*** (0.000),-0.105*** (0.001),-0.062*** (0.000),0.005 (0.396),-0.043 (0.108),0.005 (0.396)
Credit_impulse,-0.022** (0.022),-0.025** (0.012),-0.022** (0.022),-0.027*** (0.001),-0.016** (0.018),-0.027*** (0.001),-0.023 (0.115),-0.029*** (0.009),-0.023 (0.115),-0.012 (0.307),-0.014 (0.124),-0.012 (0.307),-0.014*** (0.007),-0.020*** (0.000),-0.014*** (0.007),-0.021*** (0.000),-0.013*** (0.000),-0.021*** (0.000)
Cred_structure,6.359*** (0.005),24.303*** (0.002),6.359*** (0.005),1.826** (0.040),14.690*** (0.001),1.826** (0.040),11.342*** (0.002),16.210* (0.063),11.342*** (0.002),4.751* (0.089),11.663* (0.085),4.751* (0.089),0.817 (0.109),10.144*** (0.000),0.817 (0.109),-0.572 (0.194),6.816*** (0.000),-0.572 (0.194)
Def_Zadolg_ConsCred,0.119* (0.082),0.043 (0.630),0.119* (0.082),0.122** (0.017),-0.049 (0.412),0.122** (0.017),0.189 (0.172),0.543*** (0.002),0.189 (0.172),0.250* (0.061),0.193 (0.367),0.250* (0.061),0.014 (0.495),-0.140*** (0.008),0.014 (0.495),0.058*** (0.001),-0.148*** (0.000),0.058*** (0.001)
d_Mon_Shock,-0.097*** (0.000),-0.096*** (0.000),-0.097*** (0.000),,,,-0.079 (0.135),-0.077*** (0.000),-0.079 (0.135),,,,-0.118*** (0.000),-0.119*** (0.000),-0.118*** (0.000),,,
Bonds_Rate_Correct_5Y,-1.231*** (0.000),-1.400*** (0.000),-1.231*** (0.000),0.183* (0.066),-0.198* (0.083),0.183* (0.066),-1.380*** (0.000),-1.603*** (0.000),-1.380*** (0.000),-0.558** (0.015),-0.753* (0.051),-0.558** (0.015),-0.778*** (0.000),-0.938*** (0.000),-0.778*** (0.000),0.063 (0.321),-0.183*** (0.000),0.063 (0.321)
